# hdrs

> App integration: the headers that carry the registry's assets

In [ ]:
#| default_exp hdrs

An app integrates answerui by passing its headers to `FastHTML` (or `fast_app`). However, the registry can change after that call, so the headers can't be built once at construction time; they have to be rendered fresh for each response.

In [ ]:
from fastcore.utils import *
from fasthtml.common import Style, Script
from answerui.registry import build_css
from answerui.behaviors import build_js

In [ ]:
from fasthtml.common import *
from answerui.registry import hypelt

When a FastHTML page is rendered, any object with an `__ft__` method is asked for its markup at that moment. We make use of this: `_Lazy` wraps a function to make it get called per response, and `aui_hdrs` returns the stylesheet and behavior script that way:

In [ ]:
class _Lazy:
    def __init__(self, f): self.f = f
    def __ft__(self): return self.f()

def aui_hdrs():
    "Headers for `fast_app(hdrs=...)`: the stylesheet and behavior script, rendered from the registry per response"
    return _Lazy(lambda: Style(build_css())), _Lazy(lambda: Script(build_js()))

To see the deferral, build the headers first and define a component afterwards. Rendering the stylesheet header then includes it.

In [ ]:
hdrs = aui_hdrs()
LateCard = hypelt('LateCard', css='late-card { display: block; color: teal; }', layer='components',
    on={('&', 'click'): r"el.remove()"}, doc='Defined after the app; click to remove.')
css = to_xml(hdrs[0])
print(css[css.index('/* LateCard */'):][:60])

/* LateCard */
late-card { display: block; color: teal; }
}



Redefining it changes the next render. There is no rebuild step and nothing to invalidate:

In [ ]:
LateCard = hypelt('LateCard', css='late-card { display: block; color: navy; }', layer='components', doc='Defined after the app.')
css = to_xml(hdrs[0])
assert 'navy' in css and 'teal' not in css

## A whole page

That is all an app needs. FastHTML's test client fetches a page without starting a server, and the response carries htmx, then our stylesheet and script, in the head.

In [ ]:
app = FastHTML(hdrs=aui_hdrs())
rt = app.route

@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

page = Client(app).get('/').text
print(re.sub(r'(?s)(<(style|script)>).*?(?=</\2>)', r'\1…', page))

<!doctype html>
<html>
  <head>
    <title>Hello</title>
    <link rel="canonical" href="https://testserver/">
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<script src="https://cdn.jsdelivr.net/npm/htmx.org@2.0.7/dist/htmx.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/fasthtml-js@1.0.12/fasthtml.js"></script><script src="https://cdn.jsdelivr.net/gh/answerdotai/surreal@main/surreal.js"></script><script src="https://cdn.jsdelivr.net/gh/gnat/css-scope-inline@main/script.js"></script>    <style>…</style>
<script>…</script>  </head>
  <body>
<main class="container">      <h1>Hello</h1>
<late-card>Defined after the app.</late-card></main>  </body>
</html>



`fast_app` works the same way, but it adds Pico CSS by default, so we turn that off.

In [ ]:
app, rt = fast_app(hdrs=aui_hdrs(), pico=False)
@rt
def index(): return Titled('Hello', LateCard('Defined after the app.'))

assert 'pico' not in (r:= Client(app).get('/').text)
assert 'late-card' in r

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()